# Final Paper — Overnight Sweep

Runs four experiments needed for the final paper, ordered by *information per minute* so the most critical ones finish first if Colab disconnects.

## Plan

| # | Item | Configs | Time | Why it matters |
|---|------|---------|------|----------------|
| 1 | Random-deferral baseline | 2× FID-50K | ~50 min | **Falsifiability test.** If random ≈ confidence-guided, the central claim of the paper is wrong. Highest information value. |
| 4 | Step-varying cap (static) | 3× FID-10K | ~45 min | Tests for ρ saturation above 0.5. Cheap, follows directly from the cap-binding finding. |
| 3 | Wall-clock timing | ~6× small | ~1 hr | Converts the matched-step claim into matched-wall-clock or quantifies the gap. |
| 2 | Multi-seed validation | 10× FID-50K | ~3–4 hr | Headline configs at seeds 1 and 2. Required for any quantitative significance claim. |

**Total: ~6 hours.** Run order is choices first, multi-seed last (most parallelizable across Colab sessions).

## I/O strategy

- All outputs under: `MyDrive/ARPG-assets/results/final-paper/`
- PNGs → local disk during sampling (fast). NPZs → Drive after each config (persistence).
- Resumable: re-running this notebook skips configs whose NPZ is already in Drive.

## Code changes already applied to the repo

- `models/confidence.py`: added `random_score` function and `'random'` to `CONFIDENCE_FNS`.
- `sample_c2i_ddp.py`: added `'random'` to `--confidence-metric` choices.

These let item 1 (random-deferral) reuse the existing rejection pipeline with `--confidence-metric random --rejection-threshold 2.0 --max-reject-rate 0.5`. With τ=2.0 and random scores in [0, 1), the threshold is unreachable, so the cap is always binding: exactly ⌊ρ·n_t⌋ uniformly-random positions are deferred per step (rather than depending on whether random scores happen to clear a smaller τ).

## 1. Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Runtime > Change runtime type > A100 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ARPG = "/content/drive/MyDrive/ARPG-assets"

import os
FINAL_DIR = f"{DRIVE_ARPG}/results/final-paper"
LOCAL_DIR = "/content/final-local"

# Subdirs per experiment
RANDOM_DIR    = f"{FINAL_DIR}/random-deferral"
CAP_DIR       = f"{FINAL_DIR}/cap-sweep"
WALLCLOCK_DIR = f"{FINAL_DIR}/wallclock"
MULTISEED_DIR = f"{FINAL_DIR}/multi-seed"
LOGS_DIR      = f"{FINAL_DIR}/logs"

for d in [RANDOM_DIR, CAP_DIR, WALLCLOCK_DIR, MULTISEED_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

for d in [LOCAL_DIR, f"{LOCAL_DIR}/random", f"{LOCAL_DIR}/cap",
          f"{LOCAL_DIR}/wallclock", f"{LOCAL_DIR}/multiseed", f"{LOCAL_DIR}/logs"]:
    os.makedirs(d, exist_ok=True)

print(f"Drive root: {FINAL_DIR}")
print(f"Local scratch: {LOCAL_DIR}")

In [ ]:
os.chdir('/content')
!rm -rf /content/ARPG
!git clone https://github.com/rshahbazov23/comp447-arpg-private.git /content/ARPG
%cd /content/ARPG

# Symlink heavy assets from Drive
!ln -sfn {DRIVE_ARPG}/weights weights
!ln -sfn {DRIVE_ARPG}/eval eval
!ln -sfn {DRIVE_ARPG}/external external

!pip install -q transformers einops Pillow tqdm numpy scipy tensorflow pandas

In [ ]:
# Sanity check
required = [
    "weights/arpg_300m.pt",
    "weights/vq_ds16_c2i.pt",
    "eval/VIRTUAL_imagenet256_labeled.npz",
    "external/guided-diffusion/evaluations/evaluator.py",
    "sample_c2i_ddp.py",
    "models/arpg.py",
    "models/confidence.py",
]
for p in required:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  [{status}] {p}")

# Confirm the random metric is registered
!python -c "from models.confidence import CONFIDENCE_FNS; print('Confidence metrics available:', sorted(CONFIDENCE_FNS.keys()))"

In [ ]:
# Helper: run sampler with full control over args, time it, sync NPZ to Drive.
import subprocess, time, glob, shutil

GPT_CKPT = "weights/arpg_300m.pt"
VQ_CKPT  = "weights/vq_ds16_c2i.pt"

def run_sampler(
    *,
    label,
    steps,
    num_samples,
    rejection_mode='none',
    metric='max_prob',
    tau=0.5,
    cap=0.5,
    seed=0,
    cfg_scale=5.0,
    batch_size=64,
    sample_dir_local,
    sample_dir_drive,
    log_json=None,
    skip_if_drive_npz_exists=True,
):
    """Run sample_c2i_ddp.py with the given config. Skip if the matching NPZ
    is already in `sample_dir_drive`. Returns (success, elapsed_seconds, npz_filename)."""
    os.makedirs(sample_dir_local, exist_ok=True)
    os.makedirs(sample_dir_drive, exist_ok=True)

    # Build expected NPZ filename from the same convention sample_c2i_ddp.py uses.
    # Format: ARPG-L-arpg_300m-...-step-{steps}-seed-{seed}[-mode-rejection-metric-{m}-tau-{t}-cap-{c}].npz
    if rejection_mode == 'none':
        npz_glob = f"ARPG-L-*-step-{steps}-seed-{seed}.npz"
    elif rejection_mode == 'rejection':
        npz_glob = f"ARPG-L-*-step-{steps}-seed-{seed}-mode-rejection-metric-{metric}-tau-{tau}-cap-{cap}.npz"
    else:
        npz_glob = f"ARPG-L-*-step-{steps}-seed-{seed}-mode-{rejection_mode}-*.npz"

    if skip_if_drive_npz_exists:
        existing = glob.glob(os.path.join(sample_dir_drive, npz_glob))
        if existing:
            print(f"  [SKIP] {label}: NPZ already in Drive ({os.path.basename(existing[0])})")
            return True, 0.0, os.path.basename(existing[0])

    cmd = [
        "torchrun", "--standalone", "--nnodes=1", "--nproc_per_node=1",
        "sample_c2i_ddp.py",
        "--gpt-model", "ARPG-L", "--gpt-ckpt", GPT_CKPT, "--vq-ckpt", VQ_CKPT,
        "--sample-schedule", "arccos", "--cfg-scale", str(cfg_scale),
        "--step", str(steps),
        "--num-fid-samples", str(num_samples),
        "--per-proc-batch-size", str(batch_size),
        "--sample-dir", sample_dir_local,
        "--global-seed", str(seed),
        "--no-compile",
        "--rejection-mode", rejection_mode,
    ]
    if rejection_mode == 'rejection':
        cmd += [
            "--confidence-metric", metric,
            "--rejection-threshold", str(tau),
            "--max-reject-rate", str(cap),
        ]
        if log_json:
            cmd += ["--log-json", log_json]

    print(f"  [RUN]  {label}")
    print(f"         steps={steps} samples={num_samples} mode={rejection_mode} "
          f"metric={metric} tau={tau} cap={cap} seed={seed}")
    start = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=False)
    elapsed = time.perf_counter() - start
    if proc.returncode != 0:
        print(f"  [FAIL] {label} (rc={proc.returncode}, {elapsed:.1f}s)")
        return False, elapsed, None

    # Sync NPZ to Drive
    npzs_local = sorted(glob.glob(os.path.join(sample_dir_local, "*.npz")))
    matched = [p for p in npzs_local if glob.fnmatch.fnmatch(os.path.basename(p), npz_glob)]
    if not matched:
        # Fall back: any newly created NPZ in this dir
        matched = npzs_local[-1:] if npzs_local else []
    if matched:
        src = matched[-1]
        dst = os.path.join(sample_dir_drive, os.path.basename(src))
        if not os.path.exists(dst):
            sync_start = time.perf_counter()
            shutil.copy2(src, dst)
            print(f"  [SYNC] {os.path.basename(src)} -> Drive ({time.perf_counter()-sync_start:.1f}s)")
        # Clean up the per-sample PNG folder to free disk
        npz_basename = os.path.basename(src).replace('.npz', '')
        png_dir = os.path.join(sample_dir_local, npz_basename)
        if os.path.isdir(png_dir):
            shutil.rmtree(png_dir, ignore_errors=True)
        return True, elapsed, os.path.basename(src)
    print(f"  [WARN] no NPZ found in {sample_dir_local} after run")
    return False, elapsed, None

## 2. Item 1 — Random-deferral baseline (FID-50K, 2 configs)

**Question:** If we defer ρ|Q_t| positions chosen uniformly at random instead of by lowest confidence, do we still get the FID gain? If yes → confidence ranking is doing no work. If no → confidence is the active ingredient.

**Configs:** at 16 steps and 8 steps, ρ=0.5 (matching headline rejection config), τ=2.0 so the cap is always binding (random scores in [0, 1) cannot reach τ, so the cap deterministically defers exactly ⌊ρ·n_t⌋ random positions per step).

In [ ]:
print("="*70)
print("ITEM 1 — Random-deferral baseline (FID-50K)")
print("="*70)

random_results = []
for steps in [16, 8]:
    label = f"random-deferral / step={steps} / cap=0.5 / FID-50K"
    ok, elapsed, npz = run_sampler(
        label=label,
        steps=steps,
        num_samples=50000,
        rejection_mode='rejection',
        metric='random',
        tau=2.0,            # τ > 1 with random scores in [0,1) ⇒ cap always binds ⇒ exactly ρ·n_t random positions deferred
        cap=0.5,
        seed=0,
        sample_dir_local=f"{LOCAL_DIR}/random",
        sample_dir_drive=RANDOM_DIR,
        log_json=f"{LOCAL_DIR}/logs/random-step{steps}-cap0.5.json",
    )
    random_results.append((label, ok, elapsed, npz))
    print(f"  Done in {elapsed/60:.1f} min\n")

print("\nItem 1 summary:")
for label, ok, elapsed, npz in random_results:
    status = "OK" if ok else "FAIL"
    print(f"  [{status}] {label} ({elapsed/60:.1f} min) -> {npz}")

## 3. Item 4 — Step-varying rejection cap, static extension (FID-10K, 3 configs)

**Question:** Phase 2 found cap improves FID monotonically up to ρ=0.5 with no visible saturation. Does the trend continue at ρ ∈ {0.6, 0.7, 0.8}, or do we hit a wall?

**Configs:** at 16 steps with margin/τ=0.5/ρ ∈ {0.6, 0.7, 0.8}, FID-10K (cheap pilot). If a winner clearly emerges, we can promote it to FID-50K later.

In [ ]:
print("="*70)
print("ITEM 4 — Step-varying cap sweep (FID-10K)")
print("="*70)

cap_results = []
for cap in [0.6, 0.7, 0.8]:
    label = f"cap-sweep / step=16 / margin / tau=0.5 / cap={cap} / FID-10K"
    ok, elapsed, npz = run_sampler(
        label=label,
        steps=16,
        num_samples=10000,
        rejection_mode='rejection',
        metric='margin',
        tau=0.5,
        cap=cap,
        seed=0,
        sample_dir_local=f"{LOCAL_DIR}/cap",
        sample_dir_drive=CAP_DIR,
        log_json=f"{LOCAL_DIR}/logs/cap{cap}-step16-margin.json",
    )
    cap_results.append((label, ok, elapsed, npz))
    print(f"  Done in {elapsed/60:.1f} min\n")

print("\nItem 4 summary:")
for label, ok, elapsed, npz in cap_results:
    status = "OK" if ok else "FAIL"
    print(f"  [{status}] {label} ({elapsed/60:.1f} min) -> {npz}")

## 4. Item 3 — Wall-clock timing (small, repeated runs)

**Question:** rejection uses the same number of model-forward iterations as vanilla, but the per-forward Pass-2 query batch is on average larger (deferred tokens are re-queried). Does this manifest as a measurable wall-clock difference?

**Method:** time vanilla vs. rejection (margin/τ=0.5/ρ=0.5) at 16 and 8 steps, with **2,000 samples per run** (large enough for stable timing, small enough to repeat). 3 repetitions per config to estimate variance.

Reports min/mean/std of wall-clock seconds. Skips the FID eval — these runs are throwaway for timing only.

**Note on torch.compile:** disabled in rejection mode (variable per-step accept count → dynamic shape). Vanilla *can* use torch.compile but we leave it off here for an apples-to-apples comparison.

In [ ]:
import statistics

print("="*70)
print("ITEM 3 — Wall-clock timing")
print("="*70)

WALLCLOCK_SAMPLES = 2000
WALLCLOCK_REPS = 3

wallclock_log = []
for steps in [16, 8]:
    for mode_label, kwargs in [
        ("vanilla",
         dict(rejection_mode='none', metric='max_prob', tau=0.5, cap=0.5)),
        ("rejection",
         dict(rejection_mode='rejection', metric='margin', tau=0.5, cap=0.5)),
    ]:
        times = []
        for rep in range(WALLCLOCK_REPS):
            label = f"wallclock-{mode_label} / step={steps} / rep={rep} / N={WALLCLOCK_SAMPLES}"
            # Use a unique sample_dir per rep so the existence check doesn't skip
            local_subdir = f"{LOCAL_DIR}/wallclock/rep{rep}"
            drive_subdir = f"{WALLCLOCK_DIR}/rep{rep}"
            ok, elapsed, npz = run_sampler(
                label=label,
                steps=steps,
                num_samples=WALLCLOCK_SAMPLES,
                seed=rep,    # different seed per rep so all 3 reps actually run
                sample_dir_local=local_subdir,
                sample_dir_drive=drive_subdir,
                **kwargs,
            )
            if ok and elapsed > 0:
                times.append(elapsed)
        if times:
            tmin = min(times)
            tmean = statistics.mean(times)
            tstd = statistics.pstdev(times) if len(times) > 1 else 0.0
            wallclock_log.append({
                'steps': steps, 'mode': mode_label, 'n_samples': WALLCLOCK_SAMPLES,
                'reps': len(times), 'min_s': tmin, 'mean_s': tmean, 'std_s': tstd,
            })
            print(f"  [{mode_label} / step={steps}] {len(times)} reps: "
                  f"min={tmin:.1f}s  mean={tmean:.1f}s  std={tstd:.2f}s")
        else:
            print(f"  [{mode_label} / step={steps}] no successful runs")

# Print comparison
print("\nWall-clock comparison (mean seconds for {} samples):".format(WALLCLOCK_SAMPLES))
from collections import defaultdict
by_steps = defaultdict(dict)
for row in wallclock_log:
    by_steps[row['steps']][row['mode']] = row
for steps in sorted(by_steps.keys()):
    v = by_steps[steps].get('vanilla', {}).get('mean_s')
    r = by_steps[steps].get('rejection', {}).get('mean_s')
    if v and r:
        ratio = r / v
        print(f"  step={steps}: vanilla {v:.1f}s, rejection {r:.1f}s  (ratio {ratio:.3f})")

import json
with open(f"{WALLCLOCK_DIR}/wallclock-summary.json", 'w') as f:
    json.dump(wallclock_log, f, indent=2)
print(f"\nWall-clock summary written to: {WALLCLOCK_DIR}/wallclock-summary.json")

## 5. Item 2 — Multi-seed validation (FID-50K, 10 configs)

**Question:** Are the headline FID-50K numbers stable across seeds, or are they within seed noise of vanilla?

**Configs:** vanilla and rejection at 8/16/32 steps, headline rejection config (margin/τ=0.5/ρ=0.5), with **seeds 1 and 2** (we already have seed 0 from Phase 3 — re-using it gives 3 seeds total per config).

- vanilla 8 / 16 / 32 × seeds {1, 2} = 6 runs
- rejection 8 / 16 × seeds {1, 2} = 4 runs

**This is the longest block** (~3-4 hours). Resumable — if Colab disconnects, re-running picks up where it left off.

In [ ]:
print("="*70)
print("ITEM 2 — Multi-seed validation (FID-50K)")
print("="*70)

multiseed_results = []
for seed in [1, 2]:
    seed_subdir_local = f"{LOCAL_DIR}/multiseed/seed-{seed}"
    seed_subdir_drive = f"{MULTISEED_DIR}/seed-{seed}"
    os.makedirs(seed_subdir_local, exist_ok=True)
    os.makedirs(seed_subdir_drive, exist_ok=True)

    print(f"\n--- seed {seed} ---")

    for steps in [16, 8, 32]:   # 16 first since it's the headline; 32 last (heaviest)
        label = f"multiseed-vanilla / step={steps} / seed={seed} / FID-50K"
        ok, elapsed, npz = run_sampler(
            label=label,
            steps=steps,
            num_samples=50000,
            rejection_mode='none',
            metric='max_prob', tau=0.5, cap=0.5,    # ignored when mode=none
            seed=seed,
            sample_dir_local=seed_subdir_local,
            sample_dir_drive=seed_subdir_drive,
        )
        multiseed_results.append((label, ok, elapsed, npz))
        print(f"  Done in {elapsed/60:.1f} min")

    for steps in [16, 8]:
        label = f"multiseed-rejection / step={steps} / margin / seed={seed} / FID-50K"
        ok, elapsed, npz = run_sampler(
            label=label,
            steps=steps,
            num_samples=50000,
            rejection_mode='rejection',
            metric='margin', tau=0.5, cap=0.5,
            seed=seed,
            sample_dir_local=seed_subdir_local,
            sample_dir_drive=seed_subdir_drive,
            log_json=f"{LOCAL_DIR}/logs/multiseed-step{steps}-seed{seed}.json",
        )
        multiseed_results.append((label, ok, elapsed, npz))
        print(f"  Done in {elapsed/60:.1f} min")

print("\nItem 2 summary:")
for label, ok, elapsed, npz in multiseed_results:
    status = "OK" if ok else "FAIL"
    print(f"  [{status}] {label} ({elapsed/60:.1f} min) -> {npz}")

## 6. FID evaluation

Now run the guided-diffusion evaluator on every NPZ produced above.

- **FID-50K eval per NPZ**: ~5 min (random-deferral, multi-seed)
- **FID-10K eval per NPZ**: ~1 min (cap sweep)
- Wall-clock NPZs are skipped — they were for timing, not FID.

Results written to one CSV per experiment subdir.

In [ ]:
REF_NPZ = "eval/VIRTUAL_imagenet256_labeled.npz"
EVALUATOR = "external/guided-diffusion/evaluations/evaluator.py"

def run_fid_eval_for_dir(pilot_dir, out_csv, label):
    print(f"\n--- FID eval: {label} ---")
    print(f"    pilot_dir: {pilot_dir}")
    print(f"    out_csv:   {out_csv}")
    cmd = [
        "python", "scripts/eval_pilot_sweep.py",
        "--pilot-dir", pilot_dir,
        "--reference-npz", REF_NPZ,
        "--guided-diffusion", EVALUATOR,
        "--out-csv", out_csv,
    ]
    proc = subprocess.run(cmd, capture_output=False)
    if proc.returncode != 0:
        print(f"    eval failed (rc={proc.returncode})")
        return False
    return True

run_fid_eval_for_dir(RANDOM_DIR,    f"{RANDOM_DIR}/random-deferral-results.csv", "random-deferral (FID-50K)")
run_fid_eval_for_dir(CAP_DIR,       f"{CAP_DIR}/cap-sweep-results.csv",          "cap sweep (FID-10K)")
run_fid_eval_for_dir(MULTISEED_DIR, f"{MULTISEED_DIR}/multiseed-results.csv",    "multi-seed (FID-50K)")

## 7. Summary tables

The tables you'll quote in the final paper:

1. **Random-deferral vs. confidence-guided** at 16 and 8 steps — the most important comparison.
2. **Cap saturation** at 16 steps — does ρ > 0.5 keep helping?
3. **Multi-seed mean ± std** at headline configs.
4. **Wall-clock ratios** vanilla vs. rejection.

In [ ]:
import pandas as pd
import re

def extract_steps(name):
    m = re.search(r'step-(\d+)', name)
    return int(m.group(1)) if m else None

def extract_seed(name):
    m = re.search(r'seed-(\d+)', name)
    return int(m.group(1)) if m else None

# --- Table A: random-deferral vs. confidence-guided ---
print("="*70)
print("TABLE A — Random-deferral vs. confidence-guided (FID-50K)")
print("="*70)
random_csv = f"{RANDOM_DIR}/random-deferral-results.csv"
phase3_csv = f"{DRIVE_ARPG}/results/pilot-20260421/phase3-fid50k/fid50k-results.csv"

if os.path.exists(random_csv):
    rdf = pd.read_csv(random_csv)
    rdf['steps'] = rdf['npz'].apply(extract_steps)
    rdf['source'] = 'random'
else:
    rdf = pd.DataFrame()

if os.path.exists(phase3_csv):
    pdf = pd.read_csv(phase3_csv)
    pdf['steps'] = pdf['npz'].apply(extract_steps)
    pdf['source'] = 'phase3'
else:
    pdf = pd.DataFrame()

if not rdf.empty and not pdf.empty:
    print(f"{'steps':>6} {'config':<25} {'FID-50K':>10}")
    print("-"*45)
    for steps in sorted(rdf['steps'].dropna().unique()):
        s = int(steps)
        # Vanilla baseline (Phase 3)
        van_row = pdf[(pdf['mode']=='vanilla') & (pdf['steps']==s)]
        # Confidence-guided rejection (Phase 3, margin/0.5/0.5)
        conf_row = pdf[(pdf['mode']=='rejection') & (pdf['steps']==s) &
                       (pdf['metric']=='margin') & (pdf['tau']==0.5) & (pdf['cap']==0.5)]
        # Random-deferral
        rand_row = rdf[(rdf['mode']=='rejection') & (rdf['steps']==s) &
                       (rdf['metric']=='random')]
        for src, row in [(f'vanilla', van_row), ('rejection (margin)', conf_row), ('random-deferral', rand_row)]:
            if not row.empty:
                f = row['fid'].iloc[0]
                print(f"{s:>6}  {src:<25} {f:>10.3f}")
        print()
else:
    print("  (Need both random-deferral and Phase 3 CSVs to build this table.)")

In [ ]:
# --- Table B: cap saturation at 16 steps ---
print("="*70)
print("TABLE B — Cap saturation at 16 steps, margin/tau=0.5 (FID-10K)")
print("="*70)
cap_csv = f"{CAP_DIR}/cap-sweep-results.csv"
prev_phase2 = f"{DRIVE_ARPG}/results/pilot-20260421/phase2-16step/16step-results.csv"

if os.path.exists(cap_csv):
    cdf = pd.read_csv(cap_csv)
    cdf['steps'] = cdf['npz'].apply(extract_steps)
    new_caps = cdf[(cdf['metric']=='margin') & (cdf['tau']==0.5)].sort_values('cap')
else:
    new_caps = pd.DataFrame()

if os.path.exists(prev_phase2):
    pdf2 = pd.read_csv(prev_phase2)
    pdf2['steps'] = pdf2['npz'].apply(extract_steps)
    old_caps = pdf2[(pdf2['mode']=='rejection') & (pdf2['steps']==16) &
                    (pdf2['metric']=='margin') & (pdf2['tau']==0.5)].sort_values('cap')
else:
    old_caps = pd.DataFrame()

combined = pd.concat([old_caps, new_caps], ignore_index=True).sort_values('cap')
if not combined.empty:
    print(f"{'cap':>6} {'FID-10K':>10}")
    print("-"*20)
    for _, row in combined.iterrows():
        print(f"{row['cap']:>6.2f} {row['fid']:>10.3f}")
    # Detect saturation: is FID still improving from 0.5 -> 0.6 -> 0.7 -> 0.8?
    deltas = combined.sort_values('cap')['fid'].diff().tolist()
    print(f"\nDeltas (FID improvement per cap step): {[f'{d:.3f}' if d is not None else 'NaN' for d in deltas]}")
    if all(d is None or d <= 0 for d in deltas):
        print("  Monotonic improvement — no saturation visible yet.")
    else:
        print("  Non-monotonic — saturation or noise.")
else:
    print("  (Need cap-sweep and/or Phase 2 CSVs to build this table.)")

In [ ]:
# --- Table C: multi-seed mean ± std ---
print("="*70)
print("TABLE C — Multi-seed FID-50K (seeds 0, 1, 2)")
print("="*70)
multiseed_csv = f"{MULTISEED_DIR}/multiseed-results.csv"

frames = []
if os.path.exists(multiseed_csv):
    mdf = pd.read_csv(multiseed_csv)
    mdf['steps'] = mdf['npz'].apply(extract_steps)
    mdf['seed'] = mdf['npz'].apply(extract_seed)
    frames.append(mdf)
# Add Phase 3 (seed 0)
if os.path.exists(phase3_csv):
    pdf3 = pd.read_csv(phase3_csv)
    pdf3['steps'] = pdf3['npz'].apply(extract_steps)
    pdf3['seed'] = pdf3['npz'].apply(extract_seed)
    frames.append(pdf3)

if frames:
    all_seeds = pd.concat(frames, ignore_index=True)
    all_seeds = all_seeds[all_seeds['fid'].notna()].copy()
    grouped = all_seeds.groupby(['mode', 'metric', 'tau', 'cap', 'steps'], dropna=False)['fid']
    print(f"{'config':<55} {'seeds':>8} {'mean':>10} {'std':>10}")
    print("-"*85)
    for key, group in grouped:
        mode, metric, tau, cap, steps = key
        if pd.isna(steps):
            continue
        vals = group.values
        if mode == 'vanilla':
            label = f"vanilla / step={int(steps)}"
        else:
            label = f"{mode} / step={int(steps)} / {metric} / tau={tau} / cap={cap}"
        n = len(vals)
        m = vals.mean()
        s = vals.std(ddof=0) if n > 1 else 0.0
        print(f"{label:<55} {n:>8} {m:>10.3f} {s:>10.3f}")
else:
    print("  (Need multi-seed and Phase 3 CSVs to build this table.)")

In [ ]:
# --- Table D: wall-clock comparison ---
print("="*70)
print("TABLE D — Wall-clock timing (seconds for {} samples)".format(WALLCLOCK_SAMPLES))
print("="*70)
import json
summary_path = f"{WALLCLOCK_DIR}/wallclock-summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        wcl = json.load(f)
    by_steps = {}
    for r in wcl:
        by_steps.setdefault(r['steps'], {})[r['mode']] = r
    print(f"{'steps':>6} {'vanilla(s)':>14} {'rejection(s)':>14} {'ratio':>8}")
    print("-"*50)
    for s in sorted(by_steps.keys()):
        v = by_steps[s].get('vanilla', {}).get('mean_s')
        r = by_steps[s].get('rejection', {}).get('mean_s')
        if v and r:
            print(f"{s:>6} {v:>14.1f} {r:>14.1f} {r/v:>8.3f}")
else:
    print("  (Wall-clock summary not found.)")

## 8. Done

All outputs are in `MyDrive/ARPG-assets/results/final-paper/`:

```
final-paper/
├── random-deferral/        # Item 1 NPZs + CSV
├── cap-sweep/              # Item 4 NPZs + CSV
├── wallclock/              # Item 3 NPZs + summary JSON
├── multi-seed/
│   ├── seed-1/             # Item 2 seed-1 NPZs
│   ├── seed-2/             # Item 2 seed-2 NPZs
│   └── multiseed-results.csv
└── logs/                   # rejection JSON + heatmaps
```

### What to do with these tomorrow

1. **If random-deferral ≈ confidence-guided**: serious problem. Re-frame paper claim, focus on the deferral mechanism rather than the confidence ranking. Run more random configs to verify before changing the paper's central claim.
2. **If random-deferral ≪ confidence-guided**: the paper's central claim is validated. Add Table A to the final paper as a key ablation.
3. **If cap continues improving past 0.5**: extend the cap sweep at FID-50K and add to Phase 3 results.
4. **If multi-seed std is large (>0.1 FID)**: tighten error bars before quoting any specific percentage in the abstract.
5. **If rejection wall-clock ≤ 1.05× vanilla**: drop the matched-step hedge from §6 — we have a matched-wall-clock claim.
6. **If rejection wall-clock > 1.2× vanilla**: keep the hedge, quantify the gap explicitly in §6.

Good luck.